# Risk Agent Demo

The Risk Agent is the sixth specialist, now registered alongside Finance, Sentiment, Sales,
Operations, and Growth in the boss agent's roster (`backend/agents/boss/registry.py`).

## Its 7 tools

| Tool | Computes | Data reality |
|---|---|---|
| `cancellation_trend` | Monthly cancellation rate + trend direction | Trailing-cutoff aware (reuses the Sales-agent fix) |
| `flag_payment_disputes` | Canceled orders with payment already collected | Proxy — Olist has no explicit dispute/chargeback field |
| `seller_churn_risk` | Sellers inactive for N+ months | Computed directly |
| `concentration_risk` | Customer-side revenue concentration (HHI) | Deliberately distinct from Finance's `revenue_concentration` (seller-side) — same methodology, different lens |
| `fraud_signal_detection` | Top-percentile payment-value outliers | **Switched from z-score to percentile mid-build** — see below |
| `customer_churn_prediction` | Repeat customers overdue for their next order | Heuristic (own historical gap), not a trained model. "Now" is the dataset's own cutoff, not today's date |
| `regulatory_exposure_check` | — | **Explicitly not available** — Olist has zero legal/compliance data; real exposure checking needs the (unbuilt) Compliance/HR agent's institutional documents |

## A statistical methodology bug, caught by checking the numbers rather than trusting them

The first version of `fraud_signal_detection()` used a z-score threshold (`z >= 3.0`) and
flagged **1,724 of 99,440 orders (1.73%)** as outliers. Under a normal distribution, `z >= 3`
should only catch about **0.13%** of values — over 13x fewer than what was flagged. Checking
the actual distribution explained why:

| Stat | Value |
|---|---|
| Mean | R$160.99 |
| Median | R$105.29 |
| Skewness | ~9.15 (a normal distribution is 0) |
| Max | R$13,664.08 |
| p99 | R$1,075.79 |

Per-order payment value is heavily right-skewed — mean well above median, a long tail of large
orders. Z-score assumes rough symmetry; on skewed data it systematically over-flags, because
"3 standard deviations above the mean" isn't actually rare when the mean itself is pulled up
by the tail. The fix: switch to a **percentile threshold** (top 0.5% by payment value), which
makes no assumption about distribution shape. Re-running against the same data flagged
**498 orders** — almost exactly the expected 0.5% of 99,440, confirming the method is now
behaving as intended rather than as a skewness artifact.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.agents.risk import RiskAgent
from backend.db import DatabricksClient

db = DatabricksClient()
agent = RiskAgent(db)

## Run the agent against live Delta tables

In [2]:
briefing = agent.run("What are our biggest risk exposures?")

print(f"Agent: {briefing.agent.value}")
print(f"Execution time: {briefing.execution_time_ms:.0f}ms")
print(f"Tool calls: {len(briefing.tool_calls)} ({sum(1 for t in briefing.tool_calls if t.success)} succeeded)")

Agent: Risk
Execution time: 49766ms
Tool calls: 7 (7 succeeded)


## Inspect the findings

In [3]:
for f in briefing.findings:
    print(f"[{f.severity.upper()}] (confidence {f.confidence:.2f}) {f.claim}")
    print(f"  source: {f.source}\n")

[INFO] (confidence 0.85) Cancellation rate is flat across the last 12 months
  source: cancellation_trend

[WARNING] (confidence 0.55) 625 canceled orders (143255.6 total) had payment already collected - dispute risk
  source: flag_payment_disputes

[WARNING] (confidence 0.80) 25.81% of sellers (788 of 3053) have gone inactive for 6+ months
  source: seller_churn_risk

[INFO] (confidence 0.90) Customer revenue concentration is low - top 10 customers hold 0.5% of revenue (HHI 0.3)
  source: concentration_risk

[WARNING] (confidence 0.45) 498 orders flagged as statistical payment-value outliers out of 99440 analyzed
  source: fraud_signal_detection

[WARNING] (confidence 0.50) 36.77% of repeat customers (1062 of 2888) are overdue for their next order
  source: customer_churn_prediction

[WARNING] (confidence 1.00) Regulatory exposure cannot be checked - no legal/compliance data exists in the source, and the Compliance/HR agent's institutional documents aren't built yet
  source: regulato

## Through the boss agent

Six specialists now registered. Risk naturally cross-references several others - concentration
risk vs. Finance's revenue concentration, seller churn vs. Operations' reliability data, and
cancellation trends vs. Sales' fulfillment funnel.

In [4]:
from backend.agents.boss import BossAgent

boss = BossAgent(db)
rec = boss.run("What should keep us up at night about this business?")

print(f"Agents invoked: {[a.value for a in rec.agents_invoked]}")
print(f"Overall confidence: {rec.confidence_overall}")
print(f"\nSynthesis:\n{rec.synthesis}")

if rec.dissents:
    print("\nDissents:")
    for d in rec.dissents:
        print(f"  - {[a.value for a in d.agents_involved]}: {d.summary}")

Agents invoked: ['Risk', 'Finance', 'Operations', 'Sentiment']
Overall confidence: 0.73

Synthesis:
**Board Memo: Key Risks and Areas for Immediate Attention**

**1. Revenue & Cash Flow Stability**
- Finance’s *calculate_margin_trend* shows improving contribution margin, but *detect_revenue_anomalies* flagged 13 daily revenue anomalies out of 614 days, a red flag that could indicate data integrity or fraud issues (confidence 0.85). 
- *cash_flow_forecast* projects next month’s cash‑in at $1,177,859.93 (confidence 0.55), while *refund_impact_analysis* indicates 1.68% of total payment value ($269,735.11) is at risk from canceled/unavailable orders (confidence 0.6). 
- **Action:** Investigate anomalies and strengthen real‑time monitoring; verify refund processes to protect cash flow.

**2. Cost of Goods Sold (COGS) Gap**
- Finance’s *calculate_cogs* cannot compute COGS due to missing unit cost data (confidence 1.0). This prevents accurate gross margin assessment. 
- **Action:** Source uni

## What's pending

- Compliance/HR — the last specialist. Its institutional documents (company registration, HR policy, vendor contracts) will finally give `regulatory_exposure_check` something real to check against
- `cross_reference_sla_compliance` (Compliance) pulling Operations' delivery data against contractual SLA terms is CLAUDE.md's flagship cross-agent tool — the strongest planned demo of why the multi-agent design earns its complexity